# Severity Classification with Gemini

Build severity (Low / Medium / High / Critical) is predicted at **predict time** by an LLM - it is a
completely separate model from the trained **root-cause classifier** (`rootcause_training.ipynb`).
The LLM receives a ROOT CAUSE label plus a block of evidence log lines.

**Provider: Google Gemini** via `GEMINI_API_KEY` (`https://aistudio.google.com/apikey`).
The model is read from `GEMINI_MODEL` - the deployed `.env` uses `gemini-3-flash-preview`
(`gemini-2.5-flash` is no longer available to new API keys).

**Reliability:** 429/5xx responses are retried up to 8 times, honoring Gemini's `retryDelay`
then exponential backoff. Since `gemini 3.x` is a thinking model by default, `thinkingConfig.
thinkingBudget: 0` + `maxOutputTokens: 1000` keep the reply a complete JSON object
(set `GEMINI_THINKING=1` to re-enable thinking).

**Fallback:** deterministic policy (`ml/severity_policy.json`, root-cause baselines). Callers that
do not want a misleading rule-based verdict pass `fallback_policy=False` and receive
`(None, None, reason)`; `feed_jenkins_builds.py` then leaves `llm_severity` NULL until Gemini
actually answers and the next run retries.

The cell below is the **entire live source of `ml/severity_llm.py`** inlined - keep the file and
this notebook in sync.

In [ ]:
# 2. Config (reads ../.env so the key/model match the running pipeline)
import getpass
import os
from pathlib import Path

env_path = Path("..") / ".env"  # this notebook lives in ml/
if env_path.exists():
    for line in env_path.read_text(encoding="utf-8").splitlines():
        line = line.strip()
        if line and not line.startswith("#") and "=" in line:
            k, v = line.split("=", 1)
            os.environ.setdefault(k, v)

if not os.environ.get("GEMINI_API_KEY"):
    os.environ["GEMINI_API_KEY"] = getpass.getpass("Free key from https://aistudio.google.com/apikey: ")

print(f"provider={os.environ.get('SEVERITY_LLM_PROVIDER', 'gemini')} "
      f"model={os.environ.get('GEMINI_MODEL', 'gemini-3-flash-preview')} "
      f"key_set={bool(os.environ.get('GEMINI_API_KEY'))}")

In [ ]:
# 3. Classifier - live source of ml/severity_llm.py (inlined verbatim)

import os
if "__file__" not in globals():
    globals()["__file__"] = os.path.join(os.getcwd(), "severity_llm.py")

import json
import os
import random
import re
import time
import urllib.error
import urllib.request

SEVERITY_LEVELS = ("Low", "Medium", "High", "Critical")
GEMINI_URL = "https://generativelanguage.googleapis.com/v1beta/models"
PROVIDER = os.environ.get("SEVERITY_LLM_PROVIDER", "gemini")  # gemini | none

MAX_RETRIES = 8
RETRYABLE_CODES = {429, 500, 502, 503, 504}

_POLICY_PATH = os.path.join(os.path.dirname(__file__), "severity_policy.json")
_POLICY = json.load(open(_POLICY_PATH, encoding="utf-8"))
_ROOT_CAUSE_BASELINES = _POLICY["root_cause_baselines"]
_PYLINT_MAP = _POLICY["tool_codes"]["pylint"]
_FLAKE8_MAP = _POLICY["tool_codes"]["flake8"]
_DOWNGRADE_SIGNALS = tuple(_POLICY["signals"]["downgrade"].keys())
_UPGRADE_SIGNALS = tuple(_POLICY["signals"]["upgrade"].keys())
_SEVERITY_RANK = {"Low": 0, "Medium": 1, "High": 2, "Critical": 3}

_BANDIT_SEV = re.compile(r"Severity:\s*(Low|Medium|High|Undefined)", re.IGNORECASE)
_PREFIX_RE = re.compile(r"^\s*(?:[\w./\\-]+):\d+(?::\d+)?:\s*([A-Z])\d{3,4}(?::|\s)", re.IGNORECASE)
_PYLINT_SUFFIX = re.compile(r"\([a-z0-9-]+\)\s*$", re.IGNORECASE)

_SYSTEM_PROMPT = (
    "You score the operational severity of a CI/CD build job that failed."
    " You receive the predicted ROOT CAUSE plus a block of evidence log lines."
    " Return ONLY a JSON object with two keys:\n"
    '  {"severity": "Low"|"Medium"|"High"|"Critical", "reason": "<one short sentence>"}\n'
    "Rubric:\n"
    '- Low: cosmetic/warning-level issue, build continues; e.g. lint style notes, deprecation, flaky retry.\n'
    '- Medium: partial failure or transient problem that blocks a step but not the whole system; '
    "e.g. test flake, timeout with retry, warning that needs attention.\n"
    '- High: a real failure requiring a fix; e.g. compile/test/build failure, dependency install error, '
    "deployment error, auth/permission failure.\n"
    '- Critical: outage, security breach, data loss, production impact, leaked secret, disk full, '
    "fatal/crash of the whole pipeline or service.\n"
    "When in doubt prefer the boundary that the log text itself shows."
)


def _tool_severity(line):
    m = _BANDIT_SEV.search(line)
    if m:
        s = m.group(1).capitalize()
        return 'Medium' if s == 'Undefined' else s
    if line.lstrip().startswith('>> Issue:'):
        return 'Low'
    m = _PREFIX_RE.search(line)
    if m:
        code = m.group(1).upper()
        table = _PYLINT_MAP if _PYLINT_SUFFIX.search(line) else _FLAKE8_MAP
        return table.get(code)
    return None


def policy_severity(root_cause, text):
    """Deterministic fallback: tool code > upgrade/downgrade signal > root-cause baseline.
    `text` is the flattened evidence block (or a single line)."""
    tool = _tool_severity(text)
    if tool:
        return tool
    lower = text.lower()
    baseline = _ROOT_CAUSE_BASELINES.get(root_cause, "Medium")
    if any(s in lower for s in _UPGRADE_SIGNALS):
        return "Critical"
    if any(s in lower for s in _DOWNGRADE_SIGNALS):
        lowered = "Medium" if _SEVERITY_RANK[baseline] > _SEVERITY_RANK["Medium"] else baseline
        return lowered
    return baseline


def _user_text(root_cause, evidence):
    return f"root cause: {root_cause}\n\nevidence log lines:\n{evidence}"


def _gemini_body(root_cause, evidence, force_json):
    cfg = {"temperature": 0, "maxOutputTokens": 1000}
    if force_json:
        cfg["responseMimeType"] = "application/json"
    if os.environ.get("GEMINI_THINKING", "0") != "1":
        cfg["thinkingConfig"] = {"thinkingBudget": 0}  # gemini 3.x is a thinking model by default
    return {
        "system_instruction": {"parts": [{"text": _SYSTEM_PROMPT}]},
        "contents": [{"parts": [{"text": _user_text(root_cause, evidence)}]}],
        "generationConfig": cfg,
    }


def _gemini_once(url, api_key, root_cause, evidence, timeout, force_json):
    req = urllib.request.Request(
        url + "?key=" + api_key,
        data=json.dumps(_gemini_body(root_cause, evidence, force_json)).encode("utf-8"),
        headers={"content-type": "application/json"},
        method="POST",
    )
    with urllib.request.urlopen(req, timeout=timeout) as resp:
        return json.loads(resp.read().decode("utf-8"))


def _retry_delay(e, attempt):
    """Seconds to wait before retrying: honor Gemini's retryDelay when provided."""
    try:
        body = e.read().decode("utf-8")
        delay = json.loads(body)["error"].get("retryDelay")
        if delay:
            return min(float(str(delay).rstrip("s")), 60)
    except Exception:
        pass
    return min(2 ** attempt + random.uniform(0, 1), 60)


def _call_gemini(root_cause, evidence, api_key, timeout):
    model = os.environ.get("GEMINI_MODEL", "gemini-2.5-flash")
    url = f"{GEMINI_URL}/{model}:generateContent"
    data = None
    for attempt in range(MAX_RETRIES):
        try:
            data = _gemini_once(url, api_key, root_cause, evidence, timeout, True)
            break
        except urllib.error.HTTPError as e:
            if e.code == 400:  # model may not support JSON mode -> retry once without it
                try:
                    data = _gemini_once(url, api_key, root_cause, evidence, timeout, False)
                    break
                except urllib.error.HTTPError as e2:
                    e = e2
            if e.code not in RETRYABLE_CODES or attempt == MAX_RETRIES - 1:
                raise
            time.sleep(_retry_delay(e, attempt))
    candidates = (data or {}).get("candidates", []) or []
    if not candidates:
        raise RuntimeError("no candidates: " + json.dumps(data)[:200])
    return "".join(p.get("text", "") for p in candidates[0]["content"]["parts"])


def _parse_severity(text):
    try:
        start, end = text.find("{"), text.rfind("}")
        if start != -1 and end > start:
            obj = json.loads(text[start:end + 1])
            sev = obj.get("severity")
            if isinstance(sev, str):
                sev = sev.capitalize()
            if sev in SEVERITY_LEVELS:
                return sev, obj.get("reason")
    except json.JSONDecodeError:
        pass
    m = re.search(r'"(severity)"\s*:\s*"([A-Za-z]+)"', text)
    if not m:
        return None, None
    sev = m.group(1).capitalize()
    if sev not in SEVERITY_LEVELS:
        return None, None
    rm = re.search(r'"(reason)"\s*:\s*"((?:\\.|[^"\\])*)"', text)
    return sev, rm.group(2) if rm else None


def _try_llm(root_cause, evidence, timeout):
    """Gemini, raising on total failure. Returns (severity, reason, provider)."""
    provider = PROVIDER
    if provider == "none":
        raise RuntimeError("SEVERITY_LLM_PROVIDER=none (disabled)")
    if provider != "gemini":
        raise RuntimeError(f"unsupported SEVERITY_LLM_PROVIDER={provider!r}")
    key = os.environ.get("GEMINI_API_KEY")
    if not key:
        raise RuntimeError("GEMINI_API_KEY not set")
    return *_parse_severity(_call_gemini(root_cause, evidence, key, timeout)), "gemini"


def classify_severity(root_cause, evidence, api_key=None, model=None, timeout=120, fallback_policy=True):
    """Gemini severity with deterministic policy as fallback.
    With fallback_policy=False, a failed/unparseable LLM call returns
    (None, None, reason) so callers can leave llm_severity NULL instead of
    storing a misleading rule-based verdict.
    Returns (severity, source, reason) or (None, None, reason)."""
    if api_key:
        os.environ["GEMINI_API_KEY"] = api_key
    if model:
        os.environ["GEMINI_MODEL"] = model
    try:
        sev, reason, provider = _try_llm(root_cause, evidence, timeout)
        if sev is not None:
            return sev, provider, reason or ""
        if fallback_policy:
            return policy_severity(root_cause, evidence), "policy", "unparseable LLM output"
        return None, None, "unparseable LLM output"
    except Exception as e:
        if fallback_policy:
            return policy_severity(root_cause, evidence), "policy", f"llm error: {type(e).__name__}: {e}"
        return None, None, f"llm error: {type(e).__name__}: {e}"

print("classifier defined")

In [ ]:
# 4. Demo: (root cause, evidence block) - Gemini verdict vs policy fallback
DEMOS = [
    ("FAILURE",
     "tests/test_analyzer.py::TestCalculateAverageOrderValue::test_average_basic FAILED - "
     "TypeError: unsupported operand type(s) for +: 'float' and 'NoneType'\n"
     "return round(total / len(data), 2) + None"),
    ("FAILURE",
     "ModuleNotFoundError: No module named 'psycopg2'\npip install psycopg2 failed with exit code 1"),
    ("UNSTABLE",
     "FAILED tests/ui_test.py (flaky, passed on rerun)\nElement not clickable at point (400, 210)"),
]

for rc, ev in DEMOS:
    llm_sev, llm_src, llm_reason = classify_severity(rc, ev, fallback_policy=True)
    pol_sev = policy_severity(rc, ev)
    print(f"- [{llm_sev} | {llm_src}]   [policy: {pol_sev}]   root cause={rc!r}")
    if llm_reason:
        print(f"    reason: {llm_reason.strip()}")

In [ ]:
# 5. Agreement check on an expected-severity gold set
GOLD = [
    ("FAILURE",
     "TypeError: unsupported operand type(s) for +: 'float' and 'NoneType'", "High"),
    ("FAILURE",
     "ModuleNotFoundError: No module named 'psycopg2'", "High"),
    ("FAILURE",
     "ERROR: failed to deploy gateway config to cluster, aborting deploy", "High"),
    ("UNSTABLE",
     "timeout waiting for response after 60s (request #8), retrying", "Medium"),
    ("UNSTABLE",
     "FAILED tests/ui_test.py (flaky, passed on rerun)", "Medium"),
    ("SUCCESS",
     "deprecation warning about pandas", "Low"),
    ("FAILURE",
     "fatal: unable to access repo: permission denied", "Critical"),
]

agree = 0
for rc, ev, exp in GOLD:
    sev, source, reason = classify_severity(rc, ev, fallback_policy=True)
    ok = sev == exp
    agree += ok
    print(f"  {'OK ' if ok else 'ERR'}  llm={str(sev):<8} exp={exp:<8} src={source:<6} rc={rc!r}")
print(f"\nLLM agreement with expectations: {agree}/{len(GOLD)}")

## Wiring into the pipeline
- `feed_jenkins_builds.py` calls `ml.severity_llm.classify_severity(result, log[:6000], fallback_policy=False)`
  for every build with issues or a non-SUCCESS result: the build `result` is used as the ROOT CAUSE and the
  raw log as EVIDENCE.
- It stores a verdict only for real Gemini answers (`severity is not None`); failed/unparseable calls leave
  `llm_severity` NULL so the next run retries them instead of recording a misleading policy verdict.
- **Gemini:** set `GEMINI_API_KEY` and `GEMINI_MODEL` in `.env` (deployment: `gemini-3-flash-preview`).
  `GEMINI_THINKING=1` re-enables thinking models.
- **No LLM:** `SEVERITY_LLM_PROVIDER=none` -> root-cause-baseline policy, zero external calls.